In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm


In [ ]:
NBME_PATH = "/kaggle/input/nbme-score-clinical-patient-notes"

patient_notes = pd.read_csv(f"{NBME_PATH}/patient_notes.csv")
train = pd.read_csv(f"{NBME_PATH}/train.csv")

annotated_pn_nums = train["pn_num"].unique()
annotated_notes = patient_notes[patient_notes["pn_num"].isin(annotated_pn_nums)]

annotated_notes = annotated_notes[["pn_num", "pn_history"]].drop_duplicates()
annotated_notes = annotated_notes.reset_index(drop=True)

print("Annotated notes:", annotated_notes.shape)


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()


In [ ]:
MAX_CHARS = 1200

def truncate(text):
    return text[:MAX_CHARS] if isinstance(text, str) else ""

def build_prompt(note):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a clinical information extraction system.\n"
                "Extract patient-reported symptoms or clinician-observed findings only.\n"
                "Do NOT include diagnoses, medications, procedures, labs, demographics.\n"
                "Do NOT include negated symptoms.\n"
                "Do NOT infer unstated symptoms.\n"
                "Output one symptom per line starting with '-'.\n"
                "If none, output exactly: none."
            )
        },
        {
            "role": "user",
            "content": truncate(note)
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


In [ ]:
def parse_symptoms(text):
    lines = text.splitlines()
    symptoms = []

    for line in lines:
        line = line.strip()
        if line.startswith("-"):
            s = line[1:].strip()
            if len(s) > 2:
                symptoms.append(s)

    return symptoms if symptoms else ["none"]


In [ ]:
BATCH_SIZE = 2
MAX_NEW_TOKENS = 128

results = []

for i in tqdm(range(0, len(annotated_notes), BATCH_SIZE)):
    batch = annotated_notes.iloc[i:i+BATCH_SIZE]

    prompts = [build_prompt(n) for n in batch["pn_history"]]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pn_num, out in zip(batch["pn_num"], decoded):
        parsed = parse_symptoms(out)
        results.append({
            "pn_num": pn_num,
            "model": "qwen2.5-7b-instruct",
            "predicted_symptoms": parsed
        })

    if i % 100 == 0 and i > 0:
        print(f"Processed {i} notes")


In [ ]:
OUT_DIR = "/kaggle/working/nbme_qwen_annotated"
os.makedirs(OUT_DIR, exist_ok=True)

pred_df = pd.DataFrame(results)
pred_df.to_csv(
    f"{OUT_DIR}/qwen_predictions_annotated.csv",
    index=False
)

print("Saved:", pred_df.shape)


In [1]:
import pandas as pd
import numpy as np
import re
import json
import os
import ast


In [2]:
NBME_PATH = "/kaggle/input/nbme-score-clinical-patient-notes"

train = pd.read_csv(f"{NBME_PATH}/train.csv")
features = pd.read_csv(f"{NBME_PATH}/features.csv")

print("Train:", train.shape)
print("Features:", features.shape)


Train: (14300, 6)
Features: (143, 3)


In [3]:
PRED_PATH = "/kaggle/input/cleaned/qwen_predictions_annotated_cleaned.csv"

assert os.path.exists(PRED_PATH), "Prediction file not found"

preds = pd.read_csv(PRED_PATH)

print("Predictions:", preds.shape)
preds.head()


Predictions: (1000, 3)


,pn_num,model,predicted_symptoms
0,16,qwen2.5-7b-instruct,"[""'palpitations'"", ""'chest pressure'"", ""'dizzi..."
1,41,qwen2.5-7b-instruct,"['heart pounding', 'shortness of breath', 'che..."
2,46,qwen2.5-7b-instruct,"['palpitations', 'nervousness', 'anxiousness',..."
3,82,qwen2.5-7b-instruct,['none']
4,100,qwen2.5-7b-instruct,"['Endorses light headedness, chest pressure th..."


In [4]:
feature_map = dict(zip(features["feature_num"], features["feature_text"]))
train["feature_text"] = train["feature_num"].map(feature_map)

gt_df = (
    train.groupby("pn_num")["feature_text"]
    .apply(list)
    .reset_index()
)

gt_df.head()


,pn_num,feature_text
0,16,[Family-history-of-MI-OR-Family-history-of-myo...
1,41,[Family-history-of-MI-OR-Family-history-of-myo...
2,46,[Family-history-of-MI-OR-Family-history-of-myo...
3,82,[Family-history-of-MI-OR-Family-history-of-myo...
4,100,[Family-history-of-MI-OR-Family-history-of-myo...


In [5]:
eval_df = preds.merge(gt_df, on="pn_num", how="inner")

eval_df = eval_df.rename(columns={
    "predicted_symptoms": "predicted",
    "feature_text": "gold"
})

print("Evaluation notes:", len(eval_df))
eval_df.head()


Evaluation notes: 1000


,pn_num,model,predicted,gold
0,16,qwen2.5-7b-instruct,"[""'palpitations'"", ""'chest pressure'"", ""'dizzi...",[Family-history-of-MI-OR-Family-history-of-myo...
1,41,qwen2.5-7b-instruct,"['heart pounding', 'shortness of breath', 'che...",[Family-history-of-MI-OR-Family-history-of-myo...
2,46,qwen2.5-7b-instruct,"['palpitations', 'nervousness', 'anxiousness',...",[Family-history-of-MI-OR-Family-history-of-myo...
3,82,qwen2.5-7b-instruct,['none'],[Family-history-of-MI-OR-Family-history-of-myo...
4,100,qwen2.5-7b-instruct,"['Endorses light headedness, chest pressure th...",[Family-history-of-MI-OR-Family-history-of-myo...


In [6]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_list(items):
    return [normalize_text(x) for x in items if isinstance(x, str)]


In [7]:
def is_match(pred, gold):
    return pred == gold or pred in gold or gold in pred


In [8]:
records = []

for _, row in eval_df.iterrows():
    pred_list = normalize_list(ast.literal_eval(row["predicted"]))
    gold_list = normalize_list(row["gold"])

    matched_gold = set()
    tp = 0

    for p in pred_list:
        for g in gold_list:
            if g not in matched_gold and is_match(p, g):
                matched_gold.add(g)
                tp += 1
                break

    fp = max(len(pred_list) - tp, 0)
    fn = max(len(gold_list) - tp, 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    records.append({
        "pn_num": row["pn_num"],
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

per_note_df = pd.DataFrame(records)
per_note_df.head()


,pn_num,tp,fp,fn,precision,recall,f1
0,16,1,2,12,0.333333,0.076923,0.125000
1,41,3,0,10,1.000000,0.230769,0.375000
2,46,0,5,13,0.000000,0.000000,0.000000
3,82,0,1,13,0.000000,0.000000,0.000000
4,100,1,14,12,0.066667,0.076923,0.071429


In [9]:
mean_precision = per_note_df["precision"].mean()
mean_recall = per_note_df["recall"].mean()
mean_f1 = per_note_df["f1"].mean()

print(f"Precision: {mean_precision:.4f}")
print(f"Recall:    {mean_recall:.4f}")
print(f"F1-score:  {mean_f1:.4f}")


Precision: 0.2847
Recall:    0.0948
F1-score:  0.1373


In [10]:
def bootstrap_ci(values, n_bootstrap=1000, alpha=0.05):
    rng = np.random.default_rng(42)
    samples = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        samples.append(sample.mean())
    lower = np.percentile(samples, 100 * (alpha / 2))
    upper = np.percentile(samples, 100 * (1 - alpha / 2))
    return lower, upper


prec_ci = bootstrap_ci(per_note_df["precision"].values)
rec_ci = bootstrap_ci(per_note_df["recall"].values)
f1_ci = bootstrap_ci(per_note_df["f1"].values)

print("Precision CI:", prec_ci)
print("Recall CI:", rec_ci)
print("F1 CI:", f1_ci)


Precision CI: (np.float64(0.26750040792540797), np.float64(0.299757647977023))
Recall CI: (np.float64(0.08941059703368527), np.float64(0.10031700838989444))
F1 CI: (np.float64(0.12969408849076383), np.float64(0.14494468453766224))
